# ABLATION B — DenseNet-121 + Triplet Network (no CBAM)

**Ablation Question:**
How much of the proposed model's gain comes from Triplet metric learning alone, independent of the CBAM attention mechanism?
 
**Details:**
* **Architecture:** DenseNet-121 (`baseline=True`, no CBAM) + Triplet Network
* **Training:** TripletLoss, AdamW, two-phase freeze/unfreeze (Identical to proposed model training)
* **Evaluation:** Pairwise SED on unit hypersphere (Identical to proposed model)
 
**Comparisons:**
* **Key difference from proposed:** No CBAM (`baseline=True`)
* **Key difference from baseline:** Metric learning, not classification

In [1]:
import os, sys, json, random, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from tqdm.notebook import tqdm
from PIL import Image

REPO_ROOT = os.path.abspath(os.path.join(os.path.abspath(os.getcwd()), '..'))
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from models.feature_extractor import DenseNetFeatureExtractor
from losses.triplet_loss      import TripletLoss
from utils.model_evaluation   import compute_metrics
from dataloader.tDCBAM_trainloader import get_transforms, preprocess_image, sample_augment_params

/home/lawrence/workspace/thesis/thesis/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


### STEP 1 - REPRODUCIBILITY

In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f" > [Seed] {seed}")

seed_everything(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" > [Device] {DEVICE}" +
      (f"  ({torch.cuda.get_device_name()})" if torch.cuda.is_available() else ""))

 > [Seed] 42
 > [Device] cuda  (NVIDIA GeForce RTX 5080)


### STEP 2 — CONFIGURATION

In [3]:
SPLIT_DIR      = os.path.join(REPO_ROOT, 'data', 'ratio_splits')
CHECKPOINT_DIR = os.path.join(REPO_ROOT, 'checkpoints', 'ablation_splits')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SPLIT_RATIOS = ['70_15_15']
IMG_SIZE     = 224
INPUT_SHAPE  = (IMG_SIZE, IMG_SIZE)
NUM_WORKERS  = 4

# Load dynamic configurations
CONFIG_PATH = os.path.join(REPO_ROOT, 'config', 'configs.json')
with open(CONFIG_PATH, 'r') as f:
    ALL_CONFIGS = json.load(f)

print(f" > [Ablation B] DenseNet-121 + Triplet Network — No CBAM")
print(f" > Loaded configs for: {list(ALL_CONFIGS.keys())}")

 > [Ablation B] DenseNet-121 + Triplet Network — No CBAM
 > Loaded configs for: ['cedar', 'bhsig_bengali', 'bhsig_hindi']


### STEP 3 — TRANSFORMS

In [4]:
train_transform = get_transforms(mode='train', input_shape=INPUT_SHAPE)
val_transform   = get_transforms(mode='val',   input_shape=INPUT_SHAPE)
 
print(" > [Transforms] train_transform: augmentation ON  (geometric)")
print(" > [Transforms] val_transform  : augmentation OFF (preprocessing only)")

 > [Transforms] train_transform: augmentation ON  (geometric)
 > [Transforms] val_transform  : augmentation OFF (preprocessing only)


### STEP 4 - DATASETS

In [5]:
class SplitTripletDataset(Dataset):
    def __init__(self, user_dict, input_shape=(224, 224), val_transform=None, 
                 training=True, hard_neg_ratio=0.7, silent=False):
        self.input_shape    = input_shape
        self.val_transform  = val_transform
        self.training       = training
        self.hard_neg_ratio = hard_neg_ratio

        self.user_genuine_map  = {}
        self.user_forged_map   = {}
        self.all_genuine_paths = []

        for uid, data in user_dict.items():
            gen_key  = next((k for k in data if k.lower() in ('genuine', 'gen')), None)
            forg_key = next((k for k in data if k.lower() in ('forged', 'forgeries', 'forg')), None)
            gen_paths  = data.get(gen_key,  []) if gen_key  else []
            forg_paths = data.get(forg_key, []) if forg_key else []
            if len(gen_paths) >= 2:
                self.user_genuine_map[uid] = gen_paths
                self.user_forged_map[uid]  = forg_paths
                self.all_genuine_paths.extend((p, uid) for p in gen_paths)

        self.users = list(self.user_genuine_map.keys())
        self._generate_triplets()
        
        if not silent:
            mode_label = "triplet-level aug" if training else "no aug"
            print(f"   TripletDataset: {len(self.triplets)} triplets | "
                  f"{len(self.users)} users | {mode_label}")

    def _generate_triplets(self):
        self.triplets = []
        for anchor_path, uid in self.all_genuine_paths:
            positives = [p for p in self.user_genuine_map[uid] if p != anchor_path]
            if not positives: continue
            
            pos_path  = random.choice(positives)
            forgeries = self.user_forged_map.get(uid, [])

            if random.random() < self.hard_neg_ratio and forgeries:
                neg_path = random.choice(forgeries)
            else:
                other_uid = random.choice([u for u in self.users if u != uid])
                neg_path = random.choice(self.user_genuine_map[other_uid])

            self.triplets.append((anchor_path, pos_path, neg_path))

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        a_path, p_path, n_path = self.triplets[idx]

        if self.training:
            shared_flip = random.random() < 0.5
            a_params = sample_augment_params(shared_flip=shared_flip)
            p_params = sample_augment_params(shared_flip=shared_flip)
            n_params = sample_augment_params(shared_flip=shared_flip)

            anchor   = self._load_augmented(a_path, a_params)
            positive = self._load_augmented(p_path, p_params)
            negative = self._load_augmented(n_path, n_params)
        else:
            anchor   = self._load_infer(a_path)
            positive = self._load_infer(p_path)
            negative = self._load_infer(n_path)

        return anchor, positive, negative, torch.tensor([1], dtype=torch.float32)

    def _load_augmented(self, path, augment_params):
        img = Image.open(path).convert('RGB')
        return preprocess_image(img, img_size=self.input_shape, augment=False, augment_params=augment_params)

    def _load_infer(self, path):
        img = Image.open(path).convert('RGB')
        if self.val_transform: return self.val_transform(img)
        return preprocess_image(img, img_size=self.input_shape, augment=False)


class SplitPairDataset(Dataset):
    def __init__(self, user_dict, input_shape=(224, 224), transform=None, silent=False):
        self.input_shape = input_shape
        self.transform   = transform
        self.pairs       = []

        for uid, data in user_dict.items():
            gen_key  = next((k for k in data if k.lower() in ('genuine', 'gen')), None)
            forg_key = next((k for k in data if k.lower() in ('forged', 'forgeries', 'forg')), None)
            gen_paths  = data.get(gen_key,  []) if gen_key  else []
            forg_paths = data.get(forg_key, []) if forg_key else []

            for i in range(len(gen_paths)):
                for j in range(i + 1, len(gen_paths)):
                    self.pairs.append((gen_paths[i], gen_paths[j], 1))
            for g_path in gen_paths:
                for f_path in forg_paths:
                    self.pairs.append((g_path, f_path, 0))

        if not silent:
            print(f"   PairDataset: {len(self.pairs)} pairs "
                  f"({sum(1 for _,_,l in self.pairs if l==1)} genuine, "
                  f"{sum(1 for _,_,l in self.pairs if l==0)} forged)")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        sup_path, qry_path, label = self.pairs[idx]
        return self._load(sup_path), self._load(qry_path), torch.tensor(label, dtype=torch.float32)

    def _load(self, path):
        img = Image.open(path).convert('RGB')
        if self.transform: return self.transform(img)
        return preprocess_image(img, img_size=self.input_shape, augment=False)

### STEP 5 — TRAINING & EVALUATION UTILITIES

In [6]:
def freeze_backbone(fe):
    for p in fe.get_backbone_params():
        p.requires_grad = False

def unfreeze_backbone(fe):
    for p in fe.parameters():
        p.requires_grad = True

def evaluate_model(fe, loader, device, silent=False):
    """
    Handles both validation and final evaluation using Pairwise SED.
    Uses return_curve_data=False for pure speed.
    """
    fe.eval()
    all_scores, all_labels = [], []

    with torch.no_grad():
        for sup_imgs, qry_imgs, labels in loader:
            sup_imgs = sup_imgs.to(device, non_blocking=True)
            qry_imgs = qry_imgs.to(device, non_blocking=True)
            labels   = labels.to(device, non_blocking=True)

            sup_feat  = fe(sup_imgs)
            qry_feat  = fe(qry_imgs)
            distances = torch.sum((sup_feat - qry_feat) ** 2, dim=1)
            scores    = 1.0 - (distances / 4.0)

            all_scores.extend(scores.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    metrics = compute_metrics(all_labels, all_scores, return_curve_data=False)

    if not silent:
        print(f"\n{'='*10} FINAL TEST RESULTS {'='*10}")
        for k, fmt in [('eer', ':.2%'), ('auc', ':.4f'), ('threshold', ':.4f'),
                       ('accuracy', ':.2%'), ('precision', ':.2%'),
                       ('recall', ':.2%'), ('f1', ':.2%')]:
            print(f"  {k.upper():<13}: {metrics.get(k, 0):{fmt[1:]}}")
        print("=" * 38)
        
    return metrics


def run_training(train_dataset, val_loader, device, cfg):
    """
    Ablation B Training loop. All config params are injected via `cfg` dict.
    """
    epochs         = cfg['epochs']
    phase1_epochs  = cfg['phase1_epochs']
    lr             = cfg['lr']
    margin         = cfg['margin']
    weight_decay   = cfg['weight_decay']
    batch_size     = cfg['batch_size']
    bb_lr_ratio    = cfg['backbone_lr_ratio']
    patience       = cfg['scheduler_patience']
    dataset_name   = cfg['dataset_name']
    
    VAL_EVERY = 3

    print(f"\n   {'─'*60}")
    print(f"   ABLATION B — DenseNet-121 + Triplet | {dataset_name}")
    print(f"   Epochs: {epochs} (P1 frozen: {phase1_epochs})")
    print(f"   LR: {lr} | Margin: {margin} | WD: {weight_decay} | Batch: {batch_size}")
    print(f"   CBAM: OFF | L2 Norm: ON | Loss: TripletLoss (SED)")
    print(f"   {'─'*60}")

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
        persistent_workers=(NUM_WORKERS > 0)
    )

    # baseline=True: NO CBAM. normalize=True: L2 Norm applied.
    model = DenseNetFeatureExtractor(
        backbone_name='densenet121', output_dim=1024,
        pretrained=True, baseline=True, normalize=True
    ).to(device)

    criterion = TripletLoss(margin=margin, mode='euclidean')
    scaler    = torch.amp.GradScaler('cuda')

    freeze_backbone(model)
    optimizer = optim.AdamW(model.get_head_params(), lr=lr, weight_decay=weight_decay)
    scheduler = None
    
    best_eer       = float('inf')
    best_metrics   = {}
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        if epoch == phase1_epochs:
            unfreeze_backbone(model)
            print(f"   Phase 2: Backbone unfrozen")
            optimizer = optim.AdamW([
                {'params': model.get_backbone_params(), 'lr': lr * bb_lr_ratio},
                {'params': model.get_head_params(), 'lr': lr}
            ], weight_decay=weight_decay)
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='min', factor=0.5, patience=patience, min_lr=1e-6
            )

        model.train()
        epoch_loss = 0.0

        for anchor, pos, neg, _ in tqdm(train_loader, desc=f"Train E{epoch+1:02d}", leave=False):
            anchor, pos, neg = anchor.to(device, non_blocking=True), pos.to(device, non_blocking=True), neg.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda'):
                a_emb, p_emb, n_emb = model(anchor), model(pos), model(neg)
                loss = criterion(a_emb, p_emb, n_emb)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        phase    = 1 if epoch < phase1_epochs else 2

        if (epoch + 1) % VAL_EVERY == 0 or (epoch + 1) == epochs:
            val_metrics = evaluate_model(model, val_loader, device, silent=True)
            val_eer, val_acc = val_metrics['eer'], val_metrics['accuracy']

            print(f"   [P{phase}] Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | "
                  f"Active: {criterion.last_fraction_active:.1%} | Val EER: {val_eer:.2%} | Val Acc: {val_acc:.2%}")

            if scheduler is not None: scheduler.step(val_eer)

            if val_eer < best_eer:
                best_eer, best_metrics = val_eer, val_metrics
                best_model_wts = copy.deepcopy(model.state_dict())
                print(f"   >>> Best weights updated in RAM (Val EER: {val_eer:.2%})")

        else:
            print(f"   [P{phase}] Epoch {epoch+1:02d}/{epochs} | Loss: {avg_loss:.4f} | "
                  f"Active: {criterion.last_fraction_active:.1%} | (skipping val)")

        train_dataset._generate_triplets()

    model.load_state_dict(best_model_wts)
    return model, best_metrics

### STEP 6 — RUN ALL SPLITS

In [7]:
for dataset_key, cfg in ALL_CONFIGS.items():
    DATASET_NAME = cfg['dataset_name']
    
    print(f"\n\n{'='*100}")
    print(f"{'STARTING DATASET: ' + DATASET_NAME:^100}")
    print(f"{'='*100}")
    
    all_results = {}

    for ratio in SPLIT_RATIOS:
        split_file  = os.path.join(SPLIT_DIR, f"{dataset_key}_split_{ratio}.json")
        split_label = ratio.replace('_', ':')

        if not os.path.exists(split_file):
            print(f"  SKIPPED: split file not found ({split_file})")
            continue

        with open(split_file) as f:
            split_data = json.load(f)

        train_dict = split_data['train']
        val_dict   = split_data['val']
        test_dict  = split_data['test']

        # Writer-disjoint integrity check
        assert not (set(train_dict) & set(val_dict)),  "DATA LEAK: train/val"
        assert not (set(train_dict) & set(test_dict)), "DATA LEAK: train/test"
        assert not (set(val_dict)   & set(test_dict)), "DATA LEAK: val/test"

        print(f"  Writers — Train: {len(train_dict)} | Val: {len(val_dict)} | Test: {len(test_dict)}")
        
        train_dataset = SplitTripletDataset(train_dict, input_shape=INPUT_SHAPE, val_transform=val_transform, training=True, hard_neg_ratio=cfg['hard_neg_ratio'], silent=True)
        val_dataset   = SplitPairDataset(val_dict, input_shape=INPUT_SHAPE, transform=val_transform, silent=True)
        test_dataset  = SplitPairDataset(test_dict, input_shape=INPUT_SHAPE, transform=val_transform, silent=True)

        val_loader   = DataLoader(val_dataset, batch_size=cfg['batch_size'], shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
        test_loader  = DataLoader(test_dataset, batch_size=cfg['batch_size'], shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

        seed_everything(42)
        t0 = time.time()

        trained_model, best_val_metrics = run_training(train_dataset, val_loader, DEVICE, cfg)
        t_train = time.time() - t0

        print("\n   Using best epoch weights for final test evaluation")
        final_metrics = evaluate_model(trained_model, test_loader, DEVICE, silent=False)

        key = f"{DATASET_NAME} ({split_label})"
        all_results[key] = {
            'dataset':            DATASET_NAME,
            'split':              split_label,
            'ablation':           'B — Triplet only (no CBAM)',
            'train_users':        len(train_dict),
            'val_users':          len(val_dict),
            'test_users':         len(test_dict),
            'eer':                float(final_metrics['eer']),
            'accuracy':           float(final_metrics['accuracy']),
            'auc':                float(final_metrics['auc']),
            'precision':          float(final_metrics.get('precision', 0)),
            'recall':             float(final_metrics.get('recall',    0)),
            'f1':                 float(final_metrics.get('f1',        0)),
            'train_time_seconds': round(t_train, 2),
        }

    # ── Print Summary Table for Current Dataset ───────────────────────────────────
    W = 100
    print(f"\n{'='*W}")
    print(f"{'ABLATION B — DenseNet-121 + Triplet (No CBAM) | ' + DATASET_NAME:^{W}}")
    print(f"{'='*W}")
    print(f"{'Split':<10} {'Train':<8} {'Val':<8} {'Test':<8} {'EER':>8} {'Accuracy':>10} {'AUC':>8} {'F1':>8} {'Time(s)':>10}")
    print(f"{'-'*W}")
    for key, res in all_results.items():
        print(f"{res['split']:<10} {res['train_users']:<8} {res['val_users']:<8} {res['test_users']:<8} "
              f"{res['eer']:>8.4f} {res['accuracy']:>10.4f} {res['auc']:>8.4f} {res['f1']:>8.4f} {res['train_time_seconds']:>10.2f}")
    print(f"{'='*W}")

    # Save JSON explicitly for this dataset
    results_path = os.path.join(CHECKPOINT_DIR, f'ablation_B_{dataset_key}_results.json')
    with open(results_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f"\n > Results saved → {results_path}\n")

print(f"\n{'='*100}")
print(f"{'ALL DATASETS COMPLETED SUCCESSFULLY':^100}")
print(f"{'='*100}")



                                      STARTING DATASET: CEDAR                                       
  Writers — Train: 38 | Val: 8 | Test: 9
 > [Seed] 42

   ────────────────────────────────────────────────────────────
   ABLATION B — DenseNet-121 + Triplet | CEDAR
   Epochs: 100 (P1 frozen: 8)
   LR: 0.00056 | Margin: 0.67 | WD: 1.6e-05 | Batch: 32
   CBAM: OFF | L2 Norm: ON | Loss: TripletLoss (SED)
   ────────────────────────────────────────────────────────────


Train E01:   0%|          | 0/28 [00:00<?, ?it/s]

   [P1] Epoch 01/100 | Loss: 0.4842 | Active: 81.2% | (skipping val)


Train E02:   0%|          | 0/28 [00:00<?, ?it/s]

   [P1] Epoch 02/100 | Loss: 0.4417 | Active: 62.5% | (skipping val)


...

   [P2] Epoch 98/100 | Loss: 0.0181 | Active: 0.0% | (skipping val)


Train E99:   0%|          | 0/84 [00:00<?, ?it/s]

   [P2] Epoch 99/100 | Loss: 0.0229 | Active: 0.0% | Val EER: 15.20% | Val Acc: 84.81%


Train E100:   0%|          | 0/84 [00:00<?, ?it/s]

   [P2] Epoch 100/100 | Loss: 0.0113 | Active: 0.0% | Val EER: 14.73% | Val Acc: 85.27%

   Using best epoch weights for final test evaluation

========== FINAL TEST RESULTS ==========
  EER          : 14.64%
  AUC          : 0.9260
  THRESHOLD    : 0.7870
  ACCURACY     : 85.37%
  PRECISION    : 69.10%
  RECALL       : 85.37%
  F1           : 76.38%

                    ABLATION B — DenseNet-121 + Triplet (No CBAM) | BHSig-Hindi                     
Split      Train    Val      Test          EER   Accuracy      AUC       F1    Time(s)
----------------------------------------------------------------------------------------------------
70:15:15   112      24       24         0.1464     0.8537   0.9260   0.7638    2166.57

 > Results saved → /home/lawrence/workspace/thesis/thesis/checkpoints/ablation_splits/ablation_B_bhsig_hindi_results.json


                                ALL DATASETS COMPLETED SUCCESSFULLY                                 
